In [ ]:
# One-time: the project was renamed. If an older Drive folder exists, rename it so cached data and weights are kept.
import os
try:
    from google.colab import drive; drive.mount('/content/drive')
    _old, _new = '/content/drive/MyDrive/' + 'exo' + 'lens', '/content/drive/MyDrive/anantara'
    if os.path.isdir(_old) and not os.path.exists(_new): os.rename(_old, _new); print('Renamed old Drive folder ->', _new)
except Exception: pass

# Anantara - Model A v5 (2D CNN: JWST + Hubble atmosphere molecules)
**Run:** Runtime -> GPU, then Run all (~25-30 min). Nothing to edit.

What changed vs v4:
* **Bigger data:** 40,000 simulated training spectra (was 16,000), 40 epochs (was 30), 6,000-spectrum stress test (was 2,000). Override with env vars `N_DATA`, `EPOCHS`, `N_STRESS`.
* **Higher precision target: 0.95** (was 0.90). Thresholds are tuned **on validation only**, with a +0.01 safety margin, then *verified* on the held-out test set and the stress test.
* **Leak fixed:** v4 tuned thresholds on the stress-test set and then evaluated on that same set, and the thresholds cell depended on variables created in Section 5. Thresholds now come from validation data only, and every cell runs top to bottom in order.
* **SNR precision-recall plots** moved after the stress test (they need its predictions).
* **Optional converter cell** for NASA Exoplanet Archive `.tbl` spectra before Section 6.

Raising precision costs recall - check both in Section 4/5. Say "candidate indication", never "detection".

In [ ]:
!pip -q install batman-package
import os, glob, warnings, numpy as np, pandas as pd, batman, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive'); WORK = '/content/drive/MyDrive/anantara'
except Exception:
    WORK = './anantara'
os.makedirs(WORK + '/spectra', exist_ok=True)
N_DATA = int(os.environ.get('N_DATA', 40000)); EPOCHS = int(os.environ.get('EPOCHS', 40)); N_STRESS = int(os.environ.get('N_STRESS', 6000))
TARGET_PREC = 0.95; PREC_TOL = 0.04                  # tuned on validation ONLY; test/stress precision must land within +-PREC_TOL
NL, NT = 128, 64
WL = np.linspace(0.7, 5.2, NL)
MOLS = ['H2O', 'CO2', 'CH4', 'SO2']
BANDS = {'H2O': [(0.94,.03,.3),(1.13,.04,.4),(1.38,.05,.7),(1.87,.08,.8),(2.70,.20,1.0)],
         'CO2': [(2.00,.03,.15),(2.70,.04,.3),(4.26,.07,1.0)],
         'CH4': [(1.16,.03,.2),(1.38,.03,.3),(1.68,.04,.4),(2.32,.05,.5),(3.30,.07,.9)],
         'SO2': [(4.05,.05,1.0)]}
MODES = {'jwst': (0.7, 5.2), 'nirspec': (2.87, 5.2), 'hst': (1.1, 1.7)}
MODE_P = {'jwst': 0.35, 'nirspec': 0.35, 'hst': 0.30}
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
def coverage(mode): lo, hi = MODES[mode]; return (WL >= lo) & (WL <= hi)
def vis_mask(lo, hi):   # a molecule is only scored if a strong band lies inside the observed range
    return np.array([float(any((c >= lo - s) and (c <= hi + s) and w >= 0.3 for c, s, w in BANDS[m])) for m in MOLS])
def wilson(k, n, z=1.96):
    if n == 0: return (0., 1.)
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0, c - h), min(1, c + h))
print('device', dev)

## 1. Simulator (one function; `hard=True` adds realistic nuisances)

In [ ]:
def shape_curve(rp0, a, b, P=3.0):
    rp0 = float(np.clip(rp0, 0.01, 0.3))
    inc = np.degrees(np.arccos(b / a)); T14 = P / np.pi / a * np.sqrt((1 + rp0) ** 2 - b ** 2)
    t = np.linspace(-T14, T14, NT)
    p = batman.TransitParams(); p.t0, p.per, p.rp, p.a, p.inc = 0., P, rp0, a, inc
    p.ecc, p.w, p.u, p.limb_dark = 0., 90., [0.4, 0.25], 'quadratic'
    f0 = batman.TransitModel(p, t).light_curve(p)
    return (1 - f0) / max(1 - f0.min(), 1e-9)

def to_image(depth, sig_pix, shape, cov, rng):
    flux = 1 - depth[:, None] * shape[None, :] + rng.normal(0, 1, (NL, NT)) * sig_pix[:, None]
    flux = flux / np.median(flux[:, shape < 0.02], axis=1, keepdims=True)
    img = (flux - flux[cov].mean(0)[None, :]) / sig_pix[:, None]
    img[~cov] = 0
    return img.astype(np.float32)

def smooth_noise(n, w, rng):
    k = np.exp(-0.5 * (np.arange(-3 * w, 3 * w + 1) / w) ** 2); k /= k.sum()
    z = np.convolve(rng.normal(0, 1, n + 6 * w), k, mode='same')[3 * w:3 * w + n]
    return z / (z.std() + 1e-9)

def build_spec(mode, rng, hard=True, present=None):
    rp0 = rng.uniform(0.03, 0.15); a = rng.uniform(5, 20); b = rng.uniform(0, 0.7)
    if present is None:
        present = (rng.random(4) < [0.5, 0.5, 0.5, 0.4]).astype(np.float32)
        if hard and rng.random() < 0.25: present[:] = 0              # empty atmospheres
    spec = np.zeros(NL)
    for m, pr in zip(MOLS, present):
        if pr:
            amp = 10 ** rng.uniform(np.log10(150 if hard else 400), np.log10(4000)) * 1e-6
            for c, s, w in BANDS[m]: spec += amp * w * np.exp(-0.5 * ((WL - c) / s) ** 2)
    spec *= rng.uniform(0.4 if hard else 0.5, 1.0)                    # clouds mute features
    x = (WL - WL.mean()) / np.ptp(WL)
    spec += rng.uniform(-400, 400) * 1e-6 * x + (rng.uniform(-300, 300) * 1e-6 * x ** 2 if hard else 0)
    lo, hi = MODES[mode]
    sig0 = rng.uniform(80 if hard else 100, 800 if hard else 600) * 1e-6 * (1.3 if mode == 'hst' else 1)
    if hard:
        if mode != 'hst' and rng.random() < 0.5:                      # NRS1/NRS2-like detector offset
            spec += rng.uniform(-250, 250) * 1e-6 * (WL > rng.uniform(lo + 0.3, hi - 0.3))
        for _ in range(rng.integers(0, 4)):                           # broad haze / systematics bumps
            spec += rng.uniform(-300, 300) * 1e-6 * np.exp(-0.5 * ((WL - rng.uniform(0.8, 5.0)) / rng.uniform(0.12, 0.5)) ** 2)
        spec += smooth_noise(NL, rng.integers(2, 6), rng) * sig0 * rng.uniform(0, 0.8)   # red noise
    return rp0, a, b, spec, sig0, present

def simulate(mode, rng, hard=True):
    rp0, a, b, spec, sig0, present = build_spec(mode, rng, hard)
    sig = np.full(NL, sig0) * ((1 + 0.3 * smooth_noise(NL, 8, rng)) if hard else 1)
    img = to_image(rp0 ** 2 + spec, np.abs(sig) + 1e-7, shape_curve(rp0, a, b), coverage(mode), rng)
    return img, present, vis_mask(*MODES[mode])

def make_dataset(n, seed, p_hard=0.7):
    rng = np.random.default_rng(seed); X, Y, M, H = [], [], [], []
    names, probs = list(MODE_P), list(MODE_P.values())
    for _ in range(n):
        mode = names[rng.choice(3, p=probs)]; hard = rng.random() < p_hard
        im, y, m = simulate(mode, rng, hard); X.append(im[None]); Y.append(y); M.append(m); H.append(hard)
    return np.array(X), np.array(Y), np.array(M), np.array(H)

X, Y, M, H = make_dataset(N_DATA, seed=1)
print(X.shape, 'label rates', dict(zip(MOLS, Y.mean(0).round(2))), '| empty atmospheres', float((Y.sum(1) == 0).mean().round(2)))

## 2. Train the 2D CNN (CoordConv wavelength channel, masked multi-label loss)

In [ ]:
pm = np.random.default_rng(0).permutation(len(X)); n1, n2 = int(.6 * len(X)), int(.8 * len(X))
itr, iva, ite = pm[:n1], pm[n1:n2], pm[n2:]
T = torch.tensor
Xtr, Ytr, Mtr = T(X[itr]), T(Y[itr]), T(M[itr]); Xva, Yva, Mva = T(X[iva]), T(Y[iva]), T(M[iva])
Xte, Yte, Mte, Hte = T(X[ite]), Y[ite], M[ite], H[ite]
print('train/val/test:', len(itr), len(iva), len(ite))

def blk(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
class Net2D(nn.Module):
    def __init__(s):
        super().__init__()
        s.f = nn.Sequential(blk(2, 32), blk(32, 64), blk(64, 128), nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU())
        s.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(128, 4))
    def forward(s, x):
        c = torch.linspace(-1, 1, x.shape[2], device=x.device)[None, None, :, None].expand(x.shape[0], 1, x.shape[2], x.shape[3])
        return s.fc(s.f(torch.cat([x, c], 1)).mean((2, 3)))
def mloss(model, xb, yb, mb): return (F.binary_cross_entropy_with_logits(model(xb), yb, reduction='none') * mb).sum() / mb.sum()
@torch.no_grad()
def predict(Xt):
    if len(Xt) == 0: return np.zeros((0, 4), np.float32)
    model.eval(); return torch.sigmoid(torch.cat([model(Xt[i:i + 256].to(dev)).cpu() for i in range(0, len(Xt), 256)])).numpy()

model = Net2D().to(dev); BS = 64
opt = torch.optim.AdamW(model.parameters(), 2e-3, weight_decay=1e-3)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, 4e-3, total_steps=EPOCHS * ((len(Xtr) + BS - 1) // BS))
best, best_state = 1e9, None
for ep in range(EPOCHS):
    model.train(); perm = torch.randperm(len(Xtr))
    for i in range(0, len(perm), BS):
        bi = perm[i:i + BS]; xb, yb, mb = Xtr[bi].to(dev), Ytr[bi].to(dev), Mtr[bi].to(dev)
        xb = torch.where((torch.rand(len(xb), device=dev) < 0.5)[:, None, None, None], xb.flip(-1), xb)   # mirror time axis
        loss = mloss(model, xb, yb, mb); opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    model.eval()
    with torch.no_grad(): vl = mloss(model, Xva.to(dev), Yva.to(dev), Mva.to(dev)).item()
    if vl < best: best, best_state = vl, {k: v.clone() for k, v in model.state_dict().items()}
    if ep % 5 == 4 or ep == EPOCHS - 1: print(f'epoch {ep+1}  train {loss.item():.3f}  val {vl:.3f}  best {best:.3f}')
model.load_state_dict(best_state); model.eval(); torch.save(model.state_dict(), WORK + '/modelA_v4.pt')

## 3. Thresholds: tuned on VALIDATION only for precision >= 0.95 (max recall under that constraint)

In [ ]:
Pva = predict(Xva); yva = Yva.numpy(); mva = Mva.numpy()      # validation predictions: the ONLY data used to pick thresholds

def pick_threshold(y, p, target=TARGET_PREC, margin=0.01):
    # lowest threshold (max recall) whose validation precision >= target + small safety margin (val-tuned thresholds are slightly optimistic)
    grid = np.arange(0.05, 0.995, 0.005)
    ok = [t for t in grid if (p > t).sum() >= 30 and precision_score(y, p > t, zero_division=0) >= target + margin]
    if ok: return float(min(ok)), True
    return float(max(grid, key=lambda t: precision_score(y, p > t, zero_division=0) * recall_score(y, p > t, zero_division=0))), False

THR, REACHED = np.zeros(4), []
for k, m in enumerate(MOLS):
    s = mva[:, k] > 0
    THR[k], ok = pick_threshold(yva[s, k], Pva[s, k]); REACHED.append(ok)
    pv = precision_score(yva[s, k], Pva[s, k] > THR[k], zero_division=0); rv = recall_score(yva[s, k], Pva[s, k] > THR[k], zero_division=0)
    print(f'{m}: threshold {THR[k]:.3f}  val precision {pv:.3f}  val recall {rv:.3f}' + ('' if ok else '   <-- target NOT reachable on validation, fell back to best P*R'))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

# validation predictions (Pva, yva, mva) come from the thresholds cell

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for k, m in enumerate(MOLS):
    # Filter to indices where the molecule is visible in the instrument mode
    s = mva[:, k] > 0
    y_true = yva[s, k]
    y_scores = Pva[s, k]

    precision, recall, thresholds = precision_recall_curve(y_true, y_scores)

    ax = axes[k]
    ax.plot(recall, precision, label='PR Curve', color='royalblue', lw=2)

    # Find the precision/recall corresponding to our calibrated threshold
    idx = np.argmin(np.abs(thresholds - THR[k]))
    ax.plot(recall[idx], precision[idx], 'ro', markersize=8, label=f'Threshold ({THR[k]:.2f})')

    ax.axhline(TARGET_PREC, color='gray', linestyle='--', alpha=0.5, label=f'Target Precision ({TARGET_PREC:.2f})')
    ax.set_title(f'{m} Precision-Recall Curve (Val Set)')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_xlim([0, 1.05])
    ax.set_ylim([0, 1.05])
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='lower left')

plt.tight_layout()
plt.show()

## 4. Held-out test metrics (never used for training or thresholds)

In [ ]:
Pte = predict(Xte)
def evaluate(P, Y_, V, sel, title):
    rows = []
    for k, m in enumerate(MOLS):
        s = (V[:, k] > 0) & sel
        if s.sum() < 20 or len(np.unique(Y_[s, k])) < 2: continue
        y_, p_ = Y_[s, k].astype(int), (P[s, k] > THR[k]).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_, p_, labels=[0, 1]).ravel(); lo, hi = wilson(tp, tp + fp)
        rows.append(dict(molecule=m, n=int(s.sum()), TP=tp, FP=fp, FN=fn, TN=tn, accuracy=accuracy_score(y_, p_), precision=precision_score(y_, p_, zero_division=0),
                         prec_95CI=f'{lo:.2f}-{hi:.2f}', recall=recall_score(y_, p_, zero_division=0), specificity=tn / max(tn + fp, 1), f1=f1_score(y_, p_, zero_division=0), auc=roc_auc_score(y_, P[s, k])))
    df = pd.DataFrame(rows).round(3); print(f'\n== {title} =='); print(df.to_string(index=False))
    mk = (V > 0) & sel[:, None]; yy, pp = Y_[mk].astype(int), (P[mk] > np.broadcast_to(THR, P.shape)[mk]).astype(int)
    pr = precision_score(yy, pp, zero_division=0)
    print('OVERALL  accuracy %.3f  precision %.3f  recall %.3f  F1 %.3f   %s' % (accuracy_score(yy, pp), pr, recall_score(yy, pp), f1_score(yy, pp),
          (f'precision within {TARGET_PREC:.2f} +- {PREC_TOL:.2f}' if abs(pr - TARGET_PREC) <= PREC_TOL else f'precision OUTSIDE {TARGET_PREC:.2f} +- {PREC_TOL:.2f}')))
    return df
res = evaluate(Pte, Yte, Mte, np.ones(len(Yte), bool), 'ALL held-out test')
evaluate(Pte, Yte, Mte, Hte, 'HARD held-out test only')
res.to_csv(WORK + '/modelA_v4_test_metrics.csv', index=False)
e = Yte.sum(1) == 0; fpr = ((Pte[e] > THR) & (Mte[e] > 0)).sum() / max((Mte[e] > 0).sum(), 1)
print(f'\nEMPTY atmospheres (n={e.sum()}): false-alarm rate per molecule decision = {fpr:.3f}')

## 5. Independent file-style stress test
Fresh seed (thresholds were NOT tuned on this set), generated as **(wavelength, depth, err) spectra with noise on the depth values**, then pushed through the exact same `spectrum_to_image` path used for real files. This checks the real-data pipeline, not just the simulator.

In [ ]:
def spectrum_to_image(wl, depth_ppm, err_ppm, mode, seed=0):
    rng = np.random.default_rng(seed); cov = coverage(mode) & (WL >= wl.min()) & (WL <= wl.max())
    d = np.interp(WL, wl, depth_ppm) * 1e-6; e = np.interp(WL, wl, err_ppm) * 1e-6
    sig_pix = np.maximum(e * np.sqrt(NT / 2), 1e-9)             # err is DEPTH error -> per-pixel noise
    return to_image(d, sig_pix, shape_curve(np.sqrt(max(d[cov].mean(), 1e-4)), a=10., b=0.3), cov, rng)

def synth_file(mode, rng, present):
    rp0, a, b, spec, sig0, present = build_spec(mode, rng, True, present)
    c = coverage(mode); derr = sig0 * np.sqrt(2 / NT)
    depth = rp0 ** 2 + spec + rng.normal(0, derr, NL)
    return WL[c], depth[c] * 1e6, np.full(c.sum(), derr * 1e6), present, rp0 ** 2 / derr

rng = np.random.default_rng(999); imgs, Ys, Vs, snr = [], [], [], []
# N_STRESS spectra: first half empty atmospheres, second half random mixtures
for i in range(N_STRESS):
    mode = ['jwst', 'nirspec', 'hst'][rng.choice(3, p=[.35, .35, .3])]
    pres = np.zeros(4, np.float32) if i < N_STRESS // 2 else None
    wl, d, e, pres, s = synth_file(mode, rng, pres)
    imgs.append(spectrum_to_image(wl, d, e, mode, i)); Ys.append(pres); Vs.append(vis_mask(max(wl.min(), MODES[mode][0]), min(wl.max(), MODES[mode][1]))); snr.append(s)
Ps, Ys, Vs, snr = predict(T(np.array(imgs))[:, None]), np.array(Ys), np.array(Vs), np.array(snr)
evaluate(Ps, Ys, Vs, np.ones(len(Ys), bool), f'FILE-STYLE synthetic stress test ({N_STRESS // 2} empty + {N_STRESS - N_STRESS // 2} mixed)')
pos = (Ys > 0) & (Vs > 0)
for nm, lo, hi in [('low  (<25)', 0, 25), ('mid  (25-60)', 25, 60), ('high (>60)', 60, 1e9)]:
    s = pos & ((snr >= lo) & (snr < hi))[:, None]
    if s.sum(): print(f'recall, white-light SNR {nm}: {(Ps[s] > np.broadcast_to(THR, Ps.shape)[s]).mean():.2f}  (n={int(s.sum())})')

## 5b. Precision-recall by white-light SNR (stress test)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

# Define our SNR bins
snr_bins = [('Low SNR (<25)', 0, 25), ('Mid SNR (25-60)', 25, 60), ('High SNR (>60)', 60, 1e9)]
colors = ['firebrick', 'darkorange', 'forestgreen']

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for k, m in enumerate(MOLS):
    ax = axes[k]
    # Visibility mask for current molecule
    v_mask = Vs[:, k] > 0

    for (name, lo, hi), color in zip(snr_bins, colors):
        # Filter subset by visibility and SNR bin
        bin_mask = v_mask & (snr >= lo) & (snr < hi)

        if bin_mask.sum() > 10 and len(np.unique(Ys[bin_mask, k])) > 1:
            y_true = Ys[bin_mask, k]
            y_scores = Ps[bin_mask, k]
            precision, recall, thresholds = precision_recall_curve(y_true, y_scores)

            ax.plot(recall, precision, label=f'{name} (n={bin_mask.sum()})', color=color, lw=2)

            # Mark the current threshold on the curve if available
            idx = np.argmin(np.abs(thresholds - THR[k]))
            ax.plot(recall[idx], precision[idx], marker='o', color=color, markersize=8)

    ax.axhline(TARGET_PREC, color='gray', linestyle='--', alpha=0.5, label=f'Target Precision ({TARGET_PREC:.2f})')
    ax.set_title(f'{m} Precision-Recall by SNR Bin')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_xlim([0, 1.05])
    ax.set_ylim([0, 1.05])
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='lower left')

plt.tight_layout()
plt.show()

## 6. REAL published spectra (`MyDrive/anantara/spectra/*.csv`)
Metadata tables are rejected (wavelengths must be 0.3-6 um, >= 10 rows). Edit `TRUTH` from the papers (1 reported, 0 not reported). Few decisions, same planet, different reductions -> **not independent**, so intervals are printed.

In [ ]:
# Convert NASA Exoplanet Archive .tbl spectra (downloaded into MyDrive/anantara/spectra_tbl/) into CSVs the loader understands.
# Name each file <planet>_<instrument>.tbl, e.g. wasp17b_hst_wfc3.tbl, wasp39b_jwst_prism.tbl (the loader reads planet + 'hst' from the name).
from astropy.io import ascii
SRC = WORK + '/spectra_tbl'; n_new = 0
for fn in sorted(glob.glob(SRC + '/*.tbl')):
    t = ascii.read(fn).to_pandas(); t.columns = [c.lower() for c in t.columns]
    need = ['centralwavelng', 'pl_trandep', 'pl_trandeperr1', 'pl_trandeperr2']
    if not all(c in t.columns for c in need): print('SKIPPED', os.path.basename(fn), '- columns:', list(t.columns)); continue
    out = pd.DataFrame({'wavelength_um': t['centralwavelng'], 'depth_ppm': t['pl_trandep'] * 1e4,                                   # archive reports percent
                        'depth_err_ppm': (t['pl_trandeperr1'].abs() + t['pl_trandeperr2'].abs()) / 2 * 1e4}).dropna()
    out.to_csv(WORK + '/spectra/' + os.path.basename(fn).replace('.tbl', '.csv'), index=False); n_new += 1
print(f'converted {n_new} archive spectra ->', WORK + '/spectra')

In [ ]:
BAD = ('err', 'unc', 'lim', 'upper', 'lower', 'neg', 'pos', 'flag', 'ref', 'name')
def load_spectrum(fn):
    df = pd.read_csv(fn, comment='#'); low = {c.lower(): c for c in df.columns}
    wc = next((low[c] for c in low if 'wave' in c and not any(b in c for b in BAD)), None)
    dc = next((low[c] for c in low if ('dep' in c or 'rp' in c or 'radius' in c) and not any(b in c for b in BAD)), None)
    ec = next((low[c] for c in low if ('err' in c or 'unc' in c)), None)
    if not (wc and dc and ec): return None
    num = df[[wc, dc, ec]].apply(pd.to_numeric, errors='coerce').dropna().sort_values(wc)
    if len(num) < 10: return None
    wl, d, e = num.iloc[:, 0].values, num.iloc[:, 1].values, np.abs(num.iloc[:, 2].values)
    if wl.min() < 0.3 or wl.max() > 6.0: return None
    m = np.median(d); k = 1.0 if m >= 100 else (1e4 if m >= 0.3 else 1e6)     # ppm / percent / fraction
    return wl, d * k, e * k, k

TRUTH = {'wasp39': dict(H2O=1, CO2=1, CH4=0, SO2=1), 'wasp107': dict(H2O=1, CO2=1, SO2=1),
         'hd209458': dict(H2O=1), 'hd189733': dict(H2O=1), 'wasp43': dict(H2O=1)}
out, Yr, Pr = [], [], []
for fn in sorted(glob.glob(WORK + '/spectra/*.csv')):
    nm = os.path.basename(fn); r = load_spectrum(fn)
    if r is None: print(f'SKIPPED {nm}: not a spectrum'); continue
    wl, d, e, k = r
    mode = 'hst' if ('hst' in nm.lower() or 'wfc3' in nm.lower()) else ('nirspec' if wl.min() > 2.8 else 'jwst')
    pr = predict(T(spectrum_to_image(wl, d, e, mode))[None, None])[0]
    vis = vis_mask(max(wl.min(), MODES[mode][0]), min(wl.max(), MODES[mode][1])); dec = (pr > THR).astype(int)
    print(f'{nm} [{mode}] {wl.min():.2f}-{wl.max():.2f} um  ', {m: f'{p:.2f}' + ('' if v else '*') + (' YES' if q else ' no') for m, p, v, q in zip(MOLS, pr, vis, dec)})
    out.append(dict(file=nm, mode=mode, **{m: round(float(p), 3) for m, p in zip(MOLS, pr)}))
    key = next((t for t in TRUTH if t in nm.lower().replace('-', '').replace('_', '')), None)
    for j, m in enumerate(MOLS):
        if key and m in TRUTH[key] and vis[j]: Yr.append(TRUTH[key][m]); Pr.append(dec[j])
print('(* = no strong band in this instrument range, ignored)')
pd.DataFrame(out).to_csv(WORK + '/modelA_real_predictions.csv', index=False)
if Yr:
    Yr, Pr = np.array(Yr), np.array(Pr); tp = int(((Yr == 1) & (Pr == 1)).sum()); fp = int(((Yr == 0) & (Pr == 1)).sum())
    lo, hi = wilson(tp, tp + fp)
    print(f'\nREAL: n={len(Yr)} decisions ({int((Yr==0).sum())} negatives)  accuracy {accuracy_score(Yr, Pr):.2f}  precision {precision_score(Yr, Pr, zero_division=0):.2f} (95% CI {lo:.2f}-{hi:.2f})  recall {recall_score(Yr, Pr, zero_division=0):.2f}')
    print('Too few negatives to trust precision on real data; rely on sections 4-5.')